# revit-api-rag Pipeline

数据准备阶段 — 在 Colab 中运行

## 流程
1. 克隆项目 & 安装依赖
2. 上传原始数据（API HTML + SDK 代码）
3. 解析 API 文档 → SQLite
4. 解析 SDK 代码 → SQLite
5. Embedding → ChromaDB
6. 下载生成的 .db 文件

## Step 0: 环境准备

In [ ]:
# 克隆项目
!git clone https://github.com/imkcrevit/revit-api-rag.git
%cd revit-api-rag

In [ ]:
# 安装 pipeline 依赖
!pip install -r requirements-pipeline.txt -q

In [ ]:
# 设置 API Key（在 Colab 的 Secrets 中添加，不要硬编码）
import os
from google.colab import userdata

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')  # 如果用 OpenAI

print('API Key 已设置 ✅')

In [ ]:
# 复制配置文件
!cp config/config.example.yaml config/config.yaml
print('配置文件已创建 ✅')
print('如需修改 embedding provider，请编辑 config/config.yaml')

## Step 1: 上传原始数据

In [ ]:
# 方式一：从 Google Drive 挂载
from google.colab import drive
drive.mount('/content/drive')

# 方式二：直接上传文件
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# 创建数据目录
!mkdir -p data/raw/api_html
!mkdir -p data/raw/sdk_samples
!mkdir -p data/sqlite
!mkdir -p data/chromadb

# TODO: 复制你的数据到对应目录
# 例如：
# !cp -r /content/drive/MyDrive/revit_data/html/* data/raw/api_html/
# !cp -r /content/drive/MyDrive/revit_data/sdk/* data/raw/sdk_samples/

print('数据目录已创建 ✅')
print('请将 API HTML 文件放入 data/raw/api_html/')
print('请将 SDK 代码放入 data/raw/sdk_samples/')

## Step 2: 解析 API 文档

In [ ]:
from pipeline.api_parser.parse_chm import parse_all_api_html, save_to_sqlite

# 解析 API HTML
api_data = parse_all_api_html('data/raw/api_html/')

# 存入 SQLite
save_to_sqlite(api_data, 'data/sqlite/revit_api.db')

print(f'API 解析完成：{len(api_data)} 条 ✅')

## Step 3: 解析 SDK 代码

In [ ]:
from pipeline.sdk_parser.extract import extract_all_sdk_projects, save_to_sqlite

# 提取 SDK 代码
sdk_data = extract_all_sdk_projects('data/raw/sdk_samples/')

# TODO: 代码清洗（tree-sitter + LLM）
# from pipeline.sdk_parser.clean import clean_code_with_llm
# sdk_data = clean_code_with_llm(sdk_data, config)

# 存入 SQLite
save_to_sqlite(sdk_data, 'data/sqlite/revit_sdk.db')

print(f'SDK 解析完成：{len(sdk_data)} 条 ✅')

## Step 4: Embedding 向量化

In [ ]:
from config import load_config
from pipeline.embedder.embed import embed_api_data, embed_code_data

config = load_config('config/config.yaml')
version = config.get('revit_version', '2026')

print(f'Embedding provider: {config["embedding"]["provider"]}')
print(f'Revit version: {version}')
print('开始向量化...')

In [ ]:
# API 数据向量化
embed_api_data(
    config=config,
    api_db_path='data/sqlite/revit_api.db',
    chromadb_dir=f'data/chromadb/{version}/api/',
)
print('API 向量化完成 ✅')

In [ ]:
# SDK 代码向量化
embed_code_data(
    config=config,
    sdk_db_path='data/sqlite/revit_sdk.db',
    chromadb_dir=f'data/chromadb/{version}/code/',
)
print('Code 向量化完成 ✅')

## Step 5: 验证 & 下载

In [ ]:
# 验证向量库
import chromadb
import json

for db_type in ['api', 'code']:
    db_dir = f'data/chromadb/{version}/{db_type}/'
    
    # 读取 meta.json
    with open(f'{db_dir}/meta.json') as f:
        meta = json.load(f)
    print(f'\n{db_type.upper()} 向量库:')
    print(f'  Provider: {meta["embedding_provider"]}')
    print(f'  Model: {meta["embedding_model"]}')
    print(f'  Dimension: {meta["embedding_dimension"]}')
    print(f'  Records: {meta["record_count"]}')
    
    # 测试查询
    client = chromadb.PersistentClient(path=db_dir)
    collections = client.list_collections()
    for col in collections:
        print(f'  Collection: {col.name}, Count: {col.count()}')

In [ ]:
# 打包数据文件用于下载
!tar -czf revit_rag_data.tar.gz data/
print('数据已打包: revit_rag_data.tar.gz')
print(f'文件大小: {os.path.getsize("revit_rag_data.tar.gz") / 1024 / 1024:.1f} MB')

# 下载
from google.colab import files
files.download('revit_rag_data.tar.gz')

# 或者保存到 Google Drive
# !cp revit_rag_data.tar.gz /content/drive/MyDrive/

## 完成！

下一步：
1. 将 `revit_rag_data.tar.gz` 上传到 GCP 服务器
2. 解压到项目的 `data/` 目录
3. 运行 `python -m server.app.main` 启动服务